# Diversified VLM Dataset Generator

## Causal XAI Diagnostic Pipeline — Phase 3: Synthetic Dataset Generation via Gemini

| | |
|---|---|
| **Phase** | 3 of 6 — Causal Dataset Generation |
| **Objective** | Generate diverse, natural-language diagnostic reports for XAI detection results using Gemini API |
| **Input** | `xai_results_full.json` (Phase 2 output) + 3-panel images |
| **Output** | `vlm_dataset_v2.jsonl` — instruction-tuning dataset for Qwen2-VL |
| **API** | Google Gemini 2.5 Flash Lite |

### Why Synthetic Data?

No public dataset exists that pairs XAI heatmaps with causal diagnostic explanations for
autonomous driving. This notebook bridges that gap by:

1. Reading per-object XAI metrics (Focus Score, Attention IoU, Background Leakage, Entropy, etc.)
   computed in Phase 2
2. Formatting the scene into a 3-panel image (Original | Heatmap | BBox Overlay)
3. Prompting Gemini with diverse instruction/style combinations to generate varied diagnostic text
4. Producing a JSONL dataset suitable for VLM instruction tuning

### Pipeline Position

```
Phase 1: YOLO + EigenCAM --> Phase 2: XAI Metrics --> [Phase 3: THIS NOTEBOOK] --> Phase 4: Data Integrity --> Phase 5: Fine-Tuning
```

## Table of Contents

1. [Setup & Configuration](#1-setup--configuration) — API keys, paths, sampling parameters
2. [Risk Tier Taxonomy](#2-risk-tier-taxonomy) — 29-class autonomous driving risk classification
3. [Instruction Pool Design](#3-instruction-pool-design) — 30 diverse prompt variants
4. [Prompt Style Templates](#4-prompt-style-templates) — 4 output format templates
5. [Strategic Object Sampling](#5-strategic-object-sampling) — Prioritize FP/FN, cap TP
6. [Gemini API Pipeline](#6-gemini-api-pipeline) — Rate-limited generation with resume support
7. [Output Schema](#7-output-schema) — JSONL entry structure and field descriptions

## 1. Setup & Configuration

### Dependencies

In [ ]:
!pip install google-generativeai pillow tqdm


In [ ]:
import os
import json
import time
import random
import argparse
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import google.generativeai as genai

### Configuration Parameters

| Parameter | Value | Rationale |
|---|---|---|
| `GEMINI_MODEL` | `gemini-2.5-flash-lite` | Cost-efficient for high-volume generation; sufficient quality for training data |
| `GEMINI_DELAY` | 1.2s | Safe for paid tier (~50 RPM); avoids 429 rate-limit errors |
| `MAX_RETRIES` | 3 | Exponential backoff on rate limits (30s, 60s, 90s) |
| `MAX_TP_PER_IMAGE` | 3 | Cap True Positives per image to prevent TP dominance in dataset |
| `INCLUDE_ALL_FP` | `True` | False Positives are rare and high-value — keep all |
| `INCLUDE_ALL_FN` | `True` | False Negatives are safety-critical — keep all |
| `SEED` | 42 | Reproducible sampling and style assignment |

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

GEMINI_API_KEY = "YOUR_API_KEY_HERE"  # <-- UPDATE or pass via --api-key
GEMINI_MODEL = "gemini-2.5-flash-lite"

# Rate limiting (paid tier: ~60 RPM safe)
GEMINI_DELAY = 1.2       # seconds between calls
MAX_RETRIES = 3

# Sampling strategy
MAX_TP_PER_IMAGE = 3     # cap TP objects per image
INCLUDE_ALL_FP = True    # always include all False Positives
INCLUDE_ALL_FN = True    # always include all False Negatives

SEED = 42
random.seed(SEED)

### Path Configuration

Paths auto-adjust based on environment (local vs. Kaggle):
- `xai_results_full.json` — Phase 2 output containing per-image, per-object XAI metrics
- `panels/` — Directory of 3-panel images (generated in Phase 2)
- `vlm_dataset_v2.jsonl` — Output dataset (append-mode for resume support)

In [ ]:
# ============================================================
# PATH CONFIG
# ============================================================

ENV = "local"  # "local" or "kaggle"

if ENV == "kaggle":
    BASE = "/kaggle/working/vlm_dataset"
else:
    BASE = str(Path(".").resolve().parent.parent / "vlm_dataset")

PATHS = {
    "xai_results": os.path.join(BASE, "xai_results_full.json"),
    "panels_dir": os.path.join(BASE, "panels"),
    "output_jsonl": os.path.join(BASE, "vlm_dataset_v2.jsonl"),
    "progress_file": os.path.join(BASE, "regen_progress.json"),
}

for k, v in PATHS.items():
    print(f"  {k}: {v}")

## 2. Risk Tier Taxonomy

Each of the 29 autonomous driving object classes is assigned a **risk tier** based on potential
safety impact if misdetected:

| Tier | Classes | Rationale |
|---|---|---|
| **CRITICAL** | pedestrian, cyclist, dog, stroller | Direct life-threatening if missed (FN) |
| **HIGH** | car, truck, bus, motorcycle, moped, tricycle, bicycle, construction_vehicle | Collision risk at driving speeds |
| **MODERATE** | cart, barrier, bollard, traffic_cone, traffic_island, traffic_light, traffic_sign | Navigation hazards; regulatory objects |
| **LOW** | sentry_box, debris, suitcase, dustbin, concrete_block, machinery, garbage, plastic_bag, stone, misc | Minor obstacles; low collision severity |

These tiers are injected into the Gemini prompt so the generated diagnostic text reflects
appropriate urgency levels.

In [ ]:
# ============================================================
# RISK TIERS (29 autonomous driving classes)
# ============================================================

RISK_TIERS = {
    'pedestrian': 'CRITICAL', 'cyclist': 'CRITICAL', 'dog': 'CRITICAL',
    'stroller': 'CRITICAL', 'car': 'HIGH', 'truck': 'HIGH', 'bus': 'HIGH',
    'motorcycle': 'HIGH', 'moped': 'HIGH', 'tricycle': 'HIGH',
    'bicycle': 'HIGH', 'construction_vehicle': 'HIGH', 'cart': 'MODERATE',
    'barrier': 'MODERATE', 'bollard': 'MODERATE', 'traffic_cone': 'MODERATE',
    'traffic_island': 'MODERATE', 'traffic_light': 'MODERATE',
    'traffic_sign': 'MODERATE', 'sentry_box': 'LOW', 'debris': 'LOW',
    'suitcace': 'LOW', 'dustbin': 'LOW', 'concrete_block': 'LOW',
    'machinery': 'LOW', 'garbage': 'LOW', 'plastic_bag': 'LOW',
    'stone': 'LOW', 'misc': 'LOW',
}

from collections import Counter
tier_counts = Counter(RISK_TIERS.values())
for tier in ['CRITICAL', 'HIGH', 'MODERATE', 'LOW']:
    print(f"  {tier}: {tier_counts[tier]} classes")

## 3. Instruction Pool Design

To prevent the fine-tuned VLM from overfitting to a single instruction phrasing, this pipeline
uses **30 diverse instruction variants** grouped into 5 categories:

| Category | Count | Example |
|---|---|---|
| **Role-based** | 5 | *"As a perception engineer, write a diagnostic report..."* |
| **Task-based** | 7 | *"Analyze this autonomous driving detection and explain..."* |
| **Question-based** | 6 | *"Is this detection trustworthy? Use the heatmap and XAI metrics..."* |
| **Scenario-based** | 5 | *"This detection was flagged during a routine perception audit..."* |
| **Concise/Direct** | 7 | *"Interpret the attention heatmap and metrics for this object."* |

Each training sample is **randomly assigned** one instruction from the pool, ensuring the model
generalizes across instruction formats rather than memorizing a template.

> **Design rationale:** This approach is inspired by Self-Instruct *(Wang et al., 2023)*
> and Stanford Alpaca *(Taori et al., 2023)*, which demonstrated that instruction diversity
> during training significantly improves instruction-following generalization.

In [ ]:
# ============================================================
# DIVERSE INSTRUCTION POOL (30 variants)
# ============================================================

INSTRUCTION_POOL = [
    # --- Role-based ---
    "As a perception engineer, write a diagnostic report for this detection result.",
    "You are an autonomous vehicle safety auditor. Assess this detection for deployment readiness.",
    "Acting as an XAI researcher, interpret the saliency evidence behind this detection.",
    "As a V&V (Verification & Validation) engineer, evaluate whether this detection meets safety standards.",
    "You are reviewing this detection as part of an ODD (Operational Design Domain) compliance check.",

    # --- Task-based ---
    "Analyze this autonomous driving detection and explain the model's attention behavior.",
    "Diagnose whether this object detection is reliable based on the XAI heatmap evidence.",
    "Evaluate the model's attention pattern for this detected object and assess its safety implications.",
    "Write a root-cause analysis for this detection outcome using the XAI metrics provided.",
    "Determine if this detection is driven by genuine object features or background shortcuts.",
    "Cross-reference the heatmap attention with the bounding box to verify detection integrity.",
    "Assess the quality of this object detection using the provided XAI saliency data.",

    # --- Question-based ---
    "Is this detection trustworthy? Use the heatmap and XAI metrics to justify your answer.",
    "What does the attention pattern reveal about why this object was detected or missed?",
    "Can this detection be trusted in a safety-critical autonomous driving scenario? Explain why.",
    "Does the model focus on the right features for this object, or is it relying on shortcuts?",
    "What would happen if this detection occurred in a real-world driving situation?",
    "Based on the saliency map, is the model looking at the object or the surrounding context?",

    # --- Scenario-based ---
    "This detection was flagged during a routine perception audit. Analyze the XAI evidence.",
    "A field test produced this detection result. Evaluate if it indicates a systemic model weakness.",
    "During shadow-mode testing, this result was logged. Provide your diagnostic interpretation.",
    "This case was escalated from automated QA. Review the attention metrics and explain the outcome.",
    "Imagine this detection occurs at 60 km/h in urban traffic. How critical is the model's attention pattern?",

    # --- Concise / Direct ---
    "Interpret the attention heatmap and metrics for this object.",
    "Examine the heatmap and detection metrics to determine if this result can be trusted.",
    "Review this XAI analysis of an autonomous driving detection.",
    "Provide a safety-focused analysis of this detection based on the explainability metrics.",
    "Analyze the relationship between model attention and detection outcome for this object.",
    "Explain what the saliency map tells us about this detection's reliability.",
    "Summarize the XAI diagnostic findings for this detected object.",
]

print(f"Total instruction variants: {len(INSTRUCTION_POOL)}")
print(f"\nSample instructions:")
for inst in random.sample(INSTRUCTION_POOL, 3):
    print(f"  - {inst}")

## 4. Prompt Style Templates

Beyond varying the instruction, the Gemini prompt itself uses **4 distinct output style templates**
to ensure stylistic diversity in the generated diagnostic text:

| Style | Format | Persona |
|---|---|---|
| **A: Narrative** | Free-form paragraphs, no headings | Senior AI safety researcher |
| **B: Bullet** | 5-8 concise bullet points | Perception QA engineer |
| **C: Observation-Evidence-Diagnosis** | Structured 4-section format | Autonomous driving safety auditor |
| **D: Technical** | Variable depth (brief/moderate/detailed) with random focus | XAI researcher writing a case study |

### Common Context Block

All 4 styles share the same **data injection block** that provides:
- 3-panel image description (Original | Heatmap | BBox Overlay)
- Image-level stats (TP/FP/FN counts, Precision, Recall)
- Target object metadata (class, type, risk tier, bounding box, confidence, IoU)
- Per-object XAI metrics (12 metrics)
- Global XAI metrics

In [ ]:
# ============================================================
# PROMPT CONTEXT BUILDER (shared across all styles)
# ============================================================

def _format_metrics(metrics):
    return "\n".join(f"  - {k}: {v}" for k, v in metrics.items())

def _format_globals(global_metrics):
    return "\n".join(f"  - {k}: {v}" for k, v in global_metrics.items())

def _common_context(obj, image_level):
    obj_class, obj_type = obj['class'], obj['type']
    risk = RISK_TIERS.get(obj_class, 'LOW')
    metrics = _format_metrics(obj.get('metrics', {}))
    globals_str = _format_globals(image_level.get('global_metrics', {}))
    conf_line = f"- Confidence: {obj.get('confidence', 'N/A')}" if obj_type != 'FN' else ''
    iou_line = f"- IoU with GT: {obj.get('iou', 'N/A')}" if obj_type == 'TP' else ''

    return (
        "IMAGE CONTEXT:\n"
        "This 3-panel image shows an autonomous driving scene:\n"
        "- Left: Original RGB image\n"
        "- Center: EigenCAM saliency heatmap (red/yellow = high attention)\n"
        "- Right: Heatmap overlaid with detection boxes (Green=TP, Yellow=FP, Red=FN)\n\n"
        f"IMAGE-LEVEL STATS: TP={image_level['tp_count']} | FP={image_level['fp_count']} | FN={image_level['fn_count']} | "
        f"Precision={image_level['precision']} | Recall={image_level['recall']}\n\n"
        f"TARGET OBJECT:\n"
        f"- Class: {obj_class}\n"
        f"- Detection Result: {obj_type}\n"
        f"- Risk Tier: {risk}\n"
        f"- Bounding Box (x, y, w, h): {obj.get('bbox', 'N/A')}\n"
        f"{conf_line}\n{iou_line}\n\n"
        f"PER-OBJECT XAI METRICS:\n{metrics}\n\n"
        f"GLOBAL XAI METRICS:\n{globals_str}"
    )

print("Context builder ready.")

In [ ]:
# ============================================================
# 4 PROMPT STYLE BUILDERS
# ============================================================

def build_prompt_style_a(obj, image_level):
    """Style A: Free-form narrative paragraphs, no headings."""
    ctx = _common_context(obj, image_level)
    return (
        "You are a senior AI safety researcher.\n\n"
        f"{ctx}\n\n"
        "TASK: Write a natural, flowing diagnostic narrative (2-4 paragraphs, NO headings or numbered sections). "
        "Cover: where the model's attention falls relative to this object, whether the detection relies on "
        "genuine object features or spurious background cues, what caused the detection to succeed or fail, "
        "and the safety risk level. Vary your sentence structure and vocabulary naturally. "
        "Be specific about metric values when they support your argument."
    )

def build_prompt_style_b(obj, image_level):
    """Style B: Concise bullet-point analysis."""
    ctx = _common_context(obj, image_level)
    return (
        "You are a perception QA engineer reviewing detection logs.\n\n"
        f"{ctx}\n\n"
        "TASK: Provide a concise bullet-point diagnostic (5-8 bullets). Each bullet should be 1-2 sentences. "
        "Cover attention localization, spurious correlation risk, root cause of success/failure, and safety verdict. "
        "Use plain, direct language. No headers. Start each bullet with a dash (-)."
    )

def build_prompt_style_c(obj, image_level):
    """Style C: Problem -> Evidence -> Conclusion format."""
    ctx = _common_context(obj, image_level)
    return (
        "You are an autonomous driving safety auditor.\n\n"
        f"{ctx}\n\n"
        "TASK: Structure your analysis as:\n"
        "OBSERVATION: What does the heatmap show for this object? (2-3 sentences)\n"
        "EVIDENCE: Which XAI metrics support or contradict reliable detection? (cite specific values)\n"
        "DIAGNOSIS: Why did the model succeed, fail, or false-alarm? (2-3 sentences)\n"
        "VERDICT: One-line safety risk rating with brief justification.\n\n"
        "Use these exact section labels. Be analytical and data-driven."
    )

def build_prompt_style_d(obj, image_level):
    """Style D: Technical report with varied depth."""
    ctx = _common_context(obj, image_level)
    detail_level = random.choice(["brief (3-5 sentences total)", "moderate (1-2 paragraphs)", "detailed (3-4 paragraphs)"])
    focus = random.choice([
        "Focus especially on the spurious correlation risk.",
        "Pay special attention to whether the attention pattern matches the object location.",
        "Emphasize the safety implications given the object's risk tier.",
        "Highlight what this case reveals about the model's failure modes.",
    ])
    return (
        "You are an XAI researcher writing a case study.\n\n"
        f"{ctx}\n\n"
        f"TASK: Write a {detail_level} technical analysis of this detection. "
        f"{focus} "
        "Reference specific metric values to support your claims. "
        "Use your own structure and wording - do not follow any rigid template."
    )

PROMPT_BUILDERS = [build_prompt_style_a, build_prompt_style_b, build_prompt_style_c, build_prompt_style_d]
STYLE_NAMES = ["narrative", "bullet", "observation-evidence", "technical"]

print(f"Prompt styles: {STYLE_NAMES}")

## 5. Strategic Object Sampling

### Sampling Strategy

The dataset generation prioritizes **safety-critical cases**:

| Object Type | Strategy | Rationale |
|---|---|---|
| **False Negative (FN)** | Include ALL | Most dangerous — model missed a real object |
| **False Positive (FP)** | Include ALL | Causes unnecessary braking/swerving; reveals spurious attention |
| **True Positive (TP)** | Cap at 3 per image | TPs dominate in well-performing models; uncapped would create severe class imbalance |

This produces a dataset where FP and FN cases are overrepresented relative to natural frequency,
which is intentional — the VLM needs to learn diagnostic patterns for failure cases, not just
confirm successful detections.

In [ ]:
# ============================================================
# STRATEGIC OBJECT SAMPLING
# ============================================================

def sample_objects(all_results):
    selected = []
    for result in all_results:
        panel_file = os.path.splitext(result['image'])[0] + "_3panel.jpg"
        image_level = {
            k: result[k]
            for k in ['tp_count', 'fp_count', 'fn_count', 'precision', 'recall', 'global_metrics']
        }
        tp_objects = []
        for obj in result['objects']:
            entry = (image_level, obj, panel_file)
            if obj['type'] == 'FN' and INCLUDE_ALL_FN:
                selected.append(entry)
            elif obj['type'] == 'FP' and INCLUDE_ALL_FP:
                selected.append(entry)
            elif obj['type'] == 'TP':
                tp_objects.append(entry)
        if len(tp_objects) > MAX_TP_PER_IMAGE:
            tp_objects = random.sample(tp_objects, MAX_TP_PER_IMAGE)
        selected.extend(tp_objects)
    random.shuffle(selected)
    return selected

print("Sampling function ready.")

## 6. Gemini API Pipeline

### Resume Support

The pipeline writes to the output JSONL in **append mode** and tracks completed entries
using a composite resume key: `{image_name}_{object_type}_{object_class}_{bbox}`.
On restart, it scans the existing JSONL, rebuilds the completed set, and skips already-generated entries.

### Rate Limiting & Error Handling

| Scenario | Behavior |
|---|---|
| Normal pace | 1.2s delay between calls (~50 RPM) |
| 429 Rate Limit | Exponential backoff: 30s, 60s, 90s |
| Other API errors | 10s retry delay, max 3 attempts |
| Failed entries | Logged as `[API_ERROR]` in JSONL (filtered during Phase 5 loading) |

In [ ]:
# ============================================================
# GEMINI API CALLER
# ============================================================

def call_gemini(model, image_path, prompt, max_retries=MAX_RETRIES):
    img_pil = Image.open(image_path).convert("RGB")
    for attempt in range(max_retries):
        try:
            response = model.generate_content([prompt, img_pil])
            if response.text:
                return response.text
            return "[EMPTY_RESPONSE]"
        except Exception as e:
            err_str = str(e).lower()
            if "429" in err_str or "quota" in err_str or "rate" in err_str:
                wait = (attempt + 1) * 30
                print(f"    Rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"    API error (attempt {attempt+1}): {e}")
                if attempt == max_retries - 1:
                    return f"[API_ERROR] {str(e)[:200]}"
                time.sleep(10)
    return "[API_ERROR] Max retries exceeded"

# ============================================================
# RESUME KEY UTILITIES
# ============================================================

def _normalize_bbox_for_key(bbox):
    if isinstance(bbox, str):
        try: bbox = json.loads(bbox)
        except: return bbox
    if isinstance(bbox, tuple): return list(bbox)
    return bbox

def build_resume_key(image_name, object_type, object_class, bbox):
    norm_bbox = _normalize_bbox_for_key(bbox)
    bbox_str = json.dumps(norm_bbox, ensure_ascii=False, sort_keys=True)
    return f"{image_name}_{object_type}_{object_class}_{bbox_str}"

def extract_resume_key_from_entry(entry):
    input_raw = entry.get('input', '{}')
    input_data = json.loads(input_raw) if isinstance(input_raw, str) else input_raw
    return build_resume_key(
        image_name=entry.get('image'),
        object_type=input_data.get('object_type'),
        object_class=input_data.get('object_class'),
        bbox=input_data.get('bbox'),
    )

print("API caller and resume utilities ready.")

## 7. Output Schema

Each line in the output `vlm_dataset_v2.jsonl` is a JSON object:

```json
{
  "instruction": "As a perception engineer, write a diagnostic report...",
  "input": "{\"object_class\": \"pedestrian\", \"object_type\": \"FN\", \"risk_tier\": \"CRITICAL\", ...}",
  "output": "The saliency analysis reveals a critical blind spot...",
  "image": "scene_0042_3panel.jpg",
  "style": "narrative"
}
```

### Field Descriptions

| Field | Type | Description |
|---|---|---|
| `instruction` | string | Randomly selected from 30-variant pool |
| `input` | JSON string | Structured XAI metrics + object metadata |
| `output` | string | Gemini-generated diagnostic report (training target) |
| `image` | string | Filename of 3-panel image in `panels/` |
| `style` | string | One of: `narrative`, `bullet`, `observation-evidence`, `technical` |

### Expected Dataset Statistics

| Stat | Value |
|---|---|
| Total entries | ~2,824 |
| TP / FN / FP ratio | ~60% / ~25% / ~15% |
| Output styles | ~25% each |
| Instruction variants | 30 unique |

## Usage

### As a Standalone Script

```bash
# Local
python regenerate_diverse_vlm_dataset.py --env local --api-key YOUR_KEY

# Kaggle
python regenerate_diverse_vlm_dataset.py --env kaggle --api-key YOUR_KEY

# Dry run (stats only)
python regenerate_diverse_vlm_dataset.py --env local --dry-run

# Resume with deduplication
python regenerate_diverse_vlm_dataset.py --env local --api-key YOUR_KEY --dedupe-existing
```

### As a Notebook

Run cells sequentially. Set `ENV` and `GEMINI_API_KEY` in the Configuration section,
then execute the generation pipeline.